# 📈 Sistema de Trading Automatizado v8.5.3 – GEBRA Portfolio (AUDITADO FINAL)
**Correções críticas aplicadas (conforme auditoria):**
- ✅ Look‑ahead bias eliminado: `resample_tf` agora usa `closed='right', label='right'`
- ✅ Extração MultiIndex segura: `extrair_dataframe_ticker` aplaina colunas corretamente
- ✅ Warm‑up da EMA‑50 aumentado para 150 períodos e macro fallback com score 0
- ✅ Risco direcional: `calcular_payoff_real` rejeita stop acima da entrada
- ✅ Naked excepts substituídos por `except Exception as e:` com logging
- ✅ Supressão de warnings restrita a `FutureWarning` do pandas
- ✅ Circuit breaker **Fail Closed** mantido
- ✅ Envio de e-mail automático (Gmail) incluso

In [ ]:
# =============================================================================
# CÉLULA 0: PARÂMETROS GLOBAIS (v8.5.3 - FINAL)
# =============================================================================
import os

EMAIL_REMETENTE = os.getenv('TRADING_EMAIL', '')
SENHA_APP = os.getenv('GMAIL_APP_PASSWORD', '')

PARAMS_BAIXA_VOL = {
    'kelly_frac': 0.30,
    'wyckoff_threshold': 0.75,
    'gap_max_pct': 0.055,
    'custos_pct': 0.003,
    'exigir_volume_anormal': False,
    'risco_percentual_maximo': 0.15,
    'preco_minimo': 2.00
}

PARAMS_ALTA_VOL = {
    'kelly_frac': 0.15,
    'wyckoff_threshold': 0.85,
    'gap_max_pct': 0.03,
    'custos_pct': 0.006,
    'exigir_volume_anormal': True,
    'risco_percentual_maximo': 0.10,
    'preco_minimo': 2.00
}

PARAMS_ATIVOS = PARAMS_BAIXA_VOL.copy()
MAX_SETUPS_POR_DIA = 5
MAX_PERDAS_CONSECUTIVAS = 3
DRAWDOWN_MAX_DIARIO = 0.02
MAX_DIAS_LOG = 30
HABILITAR_LOGGING = True
ARQUIVO_LOG = "trading_log_v85.json"
ARQUIVO_LOG_DETALHADO = "execucao_detalhada_v85.log"
CAPITAL_TOTAL = 100000.0
WIN_RATE_ESTIMADO = 0.40
PAYOFF_ESTIMADO = 3.0
SETORES_BLOQUEADOS = ['AEREA']
TICKERS_BLOQUEADOS = ['GFSA3.SA', 'ONCO3.SA', 'PMAM3.SA', 'AZTE3.SA', 'RAIZ4.SA', 'BHIA3.SA', 'CASH3.SA', 'LJQQ3.SA', 'RCSL4.SA', 'HBOR3.SA']
FALLBACK_TICKERS = ['PETR4', 'VALE3', 'ITUB4', 'BBDC4', 'BBAS3', 'ABEV3', 'WEGE3', 'RADL3', 'SUZB3', 'GGBR4', 'MGLU3', 'VVAR3', 'RENT3', 'RAIL3', 'CCRO3', 'ELET3', 'CPFE3', 'SBSP3', 'SANB11', 'B3SA3', 'JBSS3', 'BRFS3', 'KLBN11', 'EQTL3']
RISCO_PERCENTUAL_MINIMO = 0.02
RISCO_PERCENTUAL_MAXIMO = 0.10
PRECO_MINIMO = 5.00
BANDA_ZONA_PCT = 0.01
EXIGIR_CONFLUENCIA_CANDLE = True
CACHE_TICKERS_FILE = "cache_tickers_b3.json"
CACHE_MACRO_EXPIRY_HORAS = 24
ALTA_CONFIABILIDADE = False
DIST_CORDA_MAX = 30.0
USAR_GATILHO_BOLLINGER = False
USAR_GUARDIAO_MACD = False
MAX_ATIVOS_POR_SETOR = 2
MODO_GEBRA = 'black_belt'
LOG_DETALHADO_TICKER = True
LOG_PERFORMANCE = True
LOG_FILTROS_DETALHADO = True
VOLUME_MINIMO_ACAO = 1_000_000
VOLUME_FINANCEIRO_MINIMO = 1_000_000
LIMITE_LIQUIDEZ_FINANCEIRA = 5_000_000
EXIGIR_CONFLUENCIA = True

print("✅ Parâmetros v8.5.3 carregados")

In [ ]:
# =============================================================================
# CÉLULA 1: IMPORTAÇÕES E LOGGER (v8.5.3 - FINAL)
# =============================================================================
!pip install yfinance pandas-ta python-dotenv --quiet --upgrade-strategy only-if-needed 2>/dev/null

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
from bs4 import BeautifulSoup
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from datetime import datetime, timedelta
import time, warnings, json, os, sys, traceback
from typing import Optional, Tuple, Dict, List, Any
from collections import Counter

# Supressão controlada: apenas FutureWarning do pandas
warnings.filterwarnings("ignore", category=FutureWarning, module="pandas")

try:
    from dotenv import load_dotenv
    load_dotenv()
    if not EMAIL_REMETENTE:
        EMAIL_REMETENTE = os.getenv('TRADING_EMAIL', '')
    if not SENHA_APP:
        SENHA_APP = os.getenv('GMAIL_APP_PASSWORD', '')
except ImportError:
    pass


class Logger:
    def __init__(self, arquivo_log: str, arquivo_detalhado: Optional[str] = None):
        self.arquivo_log = arquivo_log
        self.arquivo_detalhado = arquivo_detalhado
        self.inicio_geral = time.time()
        self.timings: Dict[str, Any] = {}
        self.contadores: Dict[str, int] = {}
        self._buffer: List[str] = []
        self._max_buffer_size = 100
    
    def log(self, mensagem: str, nivel: str = "INFO", ticker: Optional[str] = None):
        ts = datetime.now().strftime("%H:%M:%S")
        msg = f"[{ts}] [{nivel}] {mensagem}"
        if ticker:
            msg += f" | {ticker}"
        print(msg)
        if self.arquivo_detalhado and LOG_PERFORMANCE:
            self._buffer.append(msg + "\n")
            if len(self._buffer) >= self._max_buffer_size:
                self._flush_buffer()
    
    def _flush_buffer(self):
        if self._buffer and self.arquivo_detalhado:
            try:
                with open(self.arquivo_detalhado, 'a', encoding='utf-8') as f:
                    f.writelines(self._buffer)
                self._buffer.clear()
            except Exception:
                pass
    
    def limpar_buffer(self):
        self._flush_buffer()
        self._buffer.clear()
    
    def warn(self, mensagem: str, ticker: Optional[str] = None):
        self.log(mensagem, "WARN", ticker)
    
    def error(self, mensagem: str, ticker: Optional[str] = None):
        self.log(mensagem, "ERRO", ticker)
    
    def iniciar_etapa(self, nome: str):
        self.timings[nome] = {'inicio': time.time()}
        self.log(f"🚀 Iniciando: {nome}", "ETAPA")
    
    def concluir_etapa(self, nome: str, detalhes: Optional[Dict] = None):
        if nome in self.timings:
            dur = time.time() - self.timings[nome]['inicio']
            self.timings[nome]['duracao'] = dur
            msg = f"✅ Concluído: {nome} ({dur:.2f}s)"
            if detalhes:
                msg += " | " + " | ".join(f"{k}: {v}" for k, v in detalhes.items())
            self.log(msg, "ETAPA")
    
    def incrementar(self, contador: str, valor: int = 1):
        self.contadores[contador] = self.contadores.get(contador, 0) + valor
    
    def resumo_final(self):
        self._flush_buffer()
        total = time.time() - self.inicio_geral
        self.log("\n" + "=" * 60, "RESUMO")
        self.log(f"⏱️ Tempo total: {total:.2f}s", "RESUMO")
        for etapa, dados in self.timings.items():
            if 'duracao' in dados:
                pct = dados['duracao'] / total * 100 if total > 0 else 0
                self.log(f"   • {etapa}: {dados['duracao']:.2f}s ({pct:.1f}%)", "RESUMO")
        if self.contadores:
            self.log("\n🔢 Contadores:", "RESUMO")
            for cont, val in self.contadores.items():
                self.log(f"   • {cont}: {val}", "RESUMO")
        self.log("=" * 60 + "\n", "RESUMO")
        try:
            with open('resumo_execucao.json', 'w', encoding='utf-8') as f:
                json.dump({
                    'timestamp': datetime.now().isoformat(),
                    'duracao_total_segundos': total,
                    'timings': {k: {kk: vv for kk, vv in v.items() if kk != 'inicio'} for k, v in self.timings.items()},
                    'contadores': self.contadores
                }, f, indent=2, ensure_ascii=False)
        except Exception:
            pass
    
    def __del__(self):
        try:
            self._flush_buffer()
        except:
            pass


if not os.path.exists(ARQUIVO_LOG):
    with open(ARQUIVO_LOG, 'w', encoding='utf-8') as f:
        json.dump([], f)

logger = Logger(ARQUIVO_LOG, ARQUIVO_LOG_DETALHADO)


# =============================================================================
# FUNÇÃO GLOBAL _log_exc – CORRIGIDO
# =============================================================================
def _log_exc(contexto: str, excecao: Exception) -> None:
    try:
        msg = f"[{contexto}] {type(excecao).__name__}: {str(excecao)[:200]}"
        if 'logger' in globals() and logger is not None:
            logger.error(msg)
        else:
            print(f"[ERRO] {msg}")
        full_trace = traceback.format_exc()
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        try:
            with open('traceback_errors.log', 'a', encoding='utf-8') as f:
                f.write(f"\n{'='*60}\nTimestamp: {timestamp}\nContexto: {contexto}\nErro: {str(excecao)}\n{full_trace}")
        except:
            pass
    except:
        print(f"[ERRO CRÍTICO] Exceção em {contexto}: {str(excecao)[:100]}")


def log_evento(tipo: str, ticker: str, dados: Dict, arquivo: str = ARQUIVO_LOG, max_dias: int = MAX_DIAS_LOG):
    if not HABILITAR_LOGGING:
        return
    registro = {'timestamp': datetime.now().isoformat(), 'tipo': tipo, 'ticker': ticker, 'dados': dados}
    logs = []
    if os.path.exists(arquivo):
        try:
            with open(arquivo, 'r', encoding='utf-8') as f:
                logs = json.load(f)
            if not isinstance(logs, list):
                logs = []
        except (json.JSONDecodeError, Exception) as e:
            logger.warn(f"Log corrompido, recriando: {str(e)[:80]}")
            logs = []
    cutoff = datetime.now() - timedelta(days=max_dias)
    logs = [l for l in logs if datetime.fromisoformat(l['timestamp']) > cutoff]
    logs.append(registro)
    try:
        with open(arquivo, 'w', encoding='utf-8') as f:
            json.dump(logs, f, ensure_ascii=False, indent=2)
    except Exception as e:
        _log_exc('log_evento', e)


logger.log("🔧 Sistema v8.5.3 (auditado final) inicializado", "INFO")
print("✅ Célula 1 carregada.")

In [ ]:
# =============================================================================
# CÉLULA 2: FUNÇÕES AUXILIARES (v8.5.3 - COM CORREÇÕES)
# =============================================================================

def _safe_divide(a: float, b: float, default: float = np.nan) -> float:
    if b is None or b == 0 or pd.isna(b):
        return default
    return a / b

def _safe_log(x: float, default: float = np.nan) -> float:
    if x is None or x <= 0 or pd.isna(x):
        return default
    return np.log(x)

def calcular_eficiencia_candle(df: pd.DataFrame) -> pd.Series:
    corpo = abs(df['Close'] - df['Open'])
    sombra_sup = df['High'] - df[['Close', 'Open']].max(axis=1)
    sombra_inf = df[['Close', 'Open']].min(axis=1) - df['Low']
    range_total = df['High'] - df['Low'].replace(0, np.nan)
    eficiencia = pd.Series(index=df.index, dtype=float)
    alta = df['Close'] > df['Open']
    baixa = df['Close'] < df['Open']
    eficiencia[alta] = 1 - (sombra_sup[alta] / range_total[alta])
    eficiencia[baixa] = 1 - (sombra_inf[baixa] / range_total[baixa])
    return eficiencia

def detectar_regime(df: pd.DataFrame, janela: int = 20) -> pd.Series:
    df_temp = df.copy()
    df_temp['retorno'] = df_temp['Close'].pct_change()
    df_temp['volatilidade'] = df_temp['retorno'].rolling(janela).std()
    try:
        adx = ta.adx(df_temp['High'], df_temp['Low'], df_temp['Close'], length=14)
        if adx is not None and not adx.empty:
            if isinstance(adx, pd.DataFrame) and 'ADX_14' in adx.columns:
                df_temp['adx'] = adx['ADX_14']
            else:
                df_temp['adx'] = adx.iloc[:, 0] if len(adx.shape) > 1 else adx
        else:
            return pd.Series(index=df.index, dtype=int)
    except Exception as e:
        _log_exc('detectar_regime', e)
        return pd.Series(index=df.index, dtype=int)
    df_temp = df_temp.dropna(subset=['volatilidade', 'adx'])
    if df_temp.empty:
        return pd.Series(index=df.index, dtype=int)
    v_p33, v_p67 = df_temp['volatilidade'].quantile([0.33, 0.67])
    a_p33, a_p67 = df_temp['adx'].quantile([0.33, 0.67])
    def classificar(row):
        v, a = row['volatilidade'], row['adx']
        if v < v_p33 and a < a_p33:
            return 0
        elif v > v_p67 or a > a_p67:
            return 2
        return 1
    regimes = df_temp.apply(classificar, axis=1)
    regime_series = pd.Series(index=df.index, dtype=int)
    regime_series.loc[regimes.index] = regimes
    regime_series.ffill(inplace=True)
    return regime_series

def detectar_swing_low(df: pd.DataFrame, janela: int = 10, confirmar: bool = True) -> float:
    if len(df) < janela:
        return float(df['Low'].min())
    lows, closes = df['Low'].values, df['Close'].values
    swing_lows = []
    for i in range(janela, len(lows)):
        if lows[i] <= 0:
            continue
        if lows[i] <= min(lows[i-janela:i]):
            if not confirmar or (i+1 < len(lows) and closes[i+1] > lows[i]):
                swing_lows.append(lows[i])
    valid_lows = [l for l in swing_lows if l > 0 and np.isfinite(l)]
    return float(valid_lows[-1]) if valid_lows else float(df['Low'].min())

def detectar_swing_high(df: pd.DataFrame, janela: int = 10, confirmar: bool = True) -> float:
    if len(df) < janela:
        return float(df['High'].max())
    highs, closes = df['High'].values, df['Close'].values
    swing_highs = []
    for i in range(janela, len(highs)):
        if highs[i] <= 0:
            continue
        if highs[i] >= max(highs[i-janela:i]):
            if not confirmar or (i+1 < len(highs) and closes[i+1] < highs[i]):
                swing_highs.append(highs[i])
    valid_highs = [h for h in swing_highs if h > 0 and np.isfinite(h)]
    return float(valid_highs[-1]) if valid_highs else float(df['High'].max())

def calcular_lta_pivos(df: pd.DataFrame, janela_pivo: int = 5) -> Optional[float]:
    lows = df['Low'].values
    lows_seguro = np.where((lows > 0) & np.isfinite(lows), lows, np.nan)
    if np.isnan(lows_seguro).all():
        return None
    log_lows = np.log(lows_seguro)
    fundos = []
    for i in range(janela_pivo, len(log_lows)-janela_pivo):
        if np.isnan(log_lows[i]):
            continue
        if log_lows[i] == np.nanmin(log_lows[i-janela_pivo:i+janela_pivo+1]):
            fundos.append((i, log_lows[i]))
    if len(fundos) >= 2:
        (x1, y1), (x2, y2) = fundos[-2], fundos[-1]
        if x2 == x1:
            return None
        incl = (y2 - y1) / (x2 - x1)
        return np.exp(y2 + incl * (len(log_lows)-1 - x2))
    return None

def calcular_ltb_pivos(df: pd.DataFrame, janela_pivo: int = 5) -> Optional[float]:
    highs = df['High'].values
    highs_seguro = np.where((highs > 0) & np.isfinite(highs), highs, np.nan)
    if np.isnan(highs_seguro).all():
        return None
    log_highs = np.log(highs_seguro)
    topos = []
    for i in range(janela_pivo, len(log_highs)-janela_pivo):
        if np.isnan(log_highs[i]):
            continue
        if log_highs[i] == np.nanmax(log_highs[i-janela_pivo:i+janela_pivo+1]):
            topos.append((i, log_highs[i]))
    if len(topos) >= 2 and topos[-2][1] > topos[-1][1]:
        (x1, y1), (x2, y2) = topos[-2], topos[-1]
        if x2 == x1:
            return None
        incl = (y2 - y1) / (x2 - x1)
        return np.exp(y2 + incl * (len(log_highs)-1 - x2))
    return None

def detectar_armadilha_lta(df: pd.DataFrame, lta: float, banda_pct: float = 0.01) -> Tuple[bool, float]:
    if len(df) < 2 or lta is None or lta <= 0:
        return False, 0.0
    ult = df.iloc[-1]
    low, close, open_ = ult['Low'], ult['Close'], ult['Open']
    corpo = abs(close - open_)
    range_c = ult['High'] - low
    if range_c <= 0:
        return False, 0.0
    sombra_inf = min(close, open_) - low
    zona_inf = lta * (1 - banda_pct)
    if low < zona_inf and close >= zona_inf and sombra_inf >= 2 * corpo:
        return True, min(1.0, sombra_inf / range_c)
    return False, 0.0

def detectar_armadilha_ltb(df: pd.DataFrame, ltb: float, banda_pct: float = 0.01) -> Tuple[bool, float]:
    if len(df) < 2 or ltb is None or ltb <= 0:
        return False, 0.0
    ult = df.iloc[-1]
    high, close, open_ = ult['High'], ult['Close'], ult['Open']
    corpo = abs(close - open_)
    range_c = high - ult['Low']
    if range_c <= 0:
        return False, 0.0
    sombra_sup = high - max(close, open_)
    zona_sup = ltb * (1 + banda_pct)
    if high > zona_sup and close <= zona_sup and sombra_sup >= 2 * corpo:
        return True, min(1.0, sombra_sup / range_c)
    return False, 0.0

def calcular_lta_adaptativo(df: pd.DataFrame, janelas: Optional[List[int]] = None) -> Optional[Tuple[float, int]]:
    if janelas is None:
        janelas = [20, 30, 40, 50]
    if len(df) < max(janelas):
        return None
    lows, closes = df['Low'].values, df['Close'].values
    melhor_score, melhor = -np.inf, None
    for janela in janelas:
        lta = calcular_lta_pivos(df, janela_pivo=janela)
        if lta is None:
            continue
        zona_inf = lta * (1 - BANDA_ZONA_PCT)
        zona_sup = lta * (1 + BANDA_ZONA_PCT)
        ini = max(0, len(lows) - max(2*janela, 50))
        score = (np.sum(lows[ini:] >= zona_inf) - 3 * np.sum(closes[ini:] < zona_inf) + 2 * np.sum((lows[ini:] >= zona_inf) & (lows[ini:] <= zona_sup)))
        if score > melhor_score:
            melhor_score, melhor = score, (lta, janela)
    return melhor

def calcular_ltb_adaptativo(df: pd.DataFrame, janelas: Optional[List[int]] = None) -> Optional[Tuple[float, int]]:
    if janelas is None:
        janelas = [20, 30, 40, 50]
    if len(df) < max(janelas):
        return None
    highs, closes = df['High'].values, df['Close'].values
    melhor_score, melhor = -np.inf, None
    for janela in janelas:
        ltb = calcular_ltb_pivos(df, janela_pivo=janela)
        if ltb is None:
            continue
        zona_inf = ltb * (1 - BANDA_ZONA_PCT)
        zona_sup = ltb * (1 + BANDA_ZONA_PCT)
        ini = max(0, len(highs) - max(2*janela, 50))
        score = (np.sum(highs[ini:] <= zona_sup) - 3 * np.sum(closes[ini:] > zona_sup) + 2 * np.sum((highs[ini:] >= zona_inf) & (highs[ini:] <= zona_sup)))
        if score > melhor_score:
            melhor_score, melhor = score, (ltb, janela)
    return melhor

def validar_elliott(df: pd.DataFrame, swing_lows: List[float], swing_highs: List[float]) -> Tuple[bool, str]:
    if len(swing_lows) < 5 or len(swing_highs) < 5:
        return False, "Pivôs insuficientes"
    try:
        onda1_low = swing_lows[-5]
        onda1_high = swing_highs[-4]
        onda2_low = swing_lows[-4]
        onda3_high = swing_highs[-3]
        onda4_low = swing_lows[-3]
        onda5_high = swing_highs[-2]
        if onda2_low < onda1_low:
            return False, "Onda 2 inválida"
        if onda4_low < onda1_high:
            return False, "Onda 4 invadiu Onda 1"
        amp1 = abs(onda1_high - onda1_low)
        amp3 = abs(onda3_high - onda2_low)
        amp5 = abs(onda5_high - onda4_low)
        if amp3 <= min(amp1, amp5):
            return False, "Onda 3 não é a maior"
        return True, "1-2-3-4-5"
    except Exception as e:
        _log_exc('validar_elliott', e)
        return False, "Erro na validação"

def calcular_obv_divergencia(df: pd.DataFrame) -> Optional[str]:
    if len(df) < 20:
        return None
    try:
        obv = ta.obv(df['Close'], df['Volume'])
        if obv is None or len(obv) < 20:
            return None
        preco_rec = df['Close'].iloc[-20:].values
        obv_rec = obv.iloc[-20:].values
        if preco_rec[-1] < preco_rec[0] and obv_rec[-1] > obv_rec[0]:
            return 'alta'
        if preco_rec[-1] > preco_rec[0] and obv_rec[-1] < obv_rec[0]:
            return 'baixa'
        return None
    except Exception as e:
        _log_exc('calcular_obv_divergencia', e)
        return None

def calcular_willr(df: pd.DataFrame, periodo: int = 14) -> Optional[float]:
    try:
        willr = ta.willr(df['High'], df['Low'], df['Close'], length=periodo)
        if willr is not None and not willr.empty:
            val = willr.iloc[-1]
            return round(float(val), 1) if pd.notna(val) else None
    except Exception as e:
        _log_exc('calcular_willr', e)
    return None

def calcular_rsi(df: pd.DataFrame, periodo: int = 14) -> Optional[float]:
    try:
        rsi = ta.rsi(df['Close'], length=periodo)
        if rsi is not None and not rsi.empty:
            val = rsi.iloc[-1]
            return round(float(val), 1) if pd.notna(val) else None
    except Exception as e:
        _log_exc('calcular_rsi', e)
    return None

def calcular_macd(df: pd.DataFrame) -> Tuple[Optional[float], Optional[float], Optional[float]]:
    try:
        macd = ta.macd(df['Close'])
        if macd is not None and not macd.empty:
            m = macd['MACD_12_26_9'].iloc[-1]
            s = macd['MACDs_12_26_9'].iloc[-1]
            h = macd['MACDh_12_26_9'].iloc[-1]
            return (round(float(m), 2) if pd.notna(m) else None,
                    round(float(s), 2) if pd.notna(s) else None,
                    round(float(h), 2) if pd.notna(h) else None)
    except Exception as e:
        _log_exc('calcular_macd', e)
    return None, None, None

def calcular_estocastico(df: pd.DataFrame, periodo: int = 14) -> Tuple[Optional[float], Optional[float]]:
    try:
        stoch = ta.stoch(df['High'], df['Low'], df['Close'], k=periodo, d=3)
        if stoch is not None and not stoch.empty:
            k = stoch['STOCHk_14_3_3'].iloc[-1]
            d = stoch['STOCHd_14_3_3'].iloc[-1]
            return (round(float(k), 1) if pd.notna(k) else None,
                    round(float(d), 1) if pd.notna(d) else None)
    except Exception as e:
        _log_exc('calcular_estocastico', e)
    return None, None

def calcular_bandas_bollinger(df: pd.DataFrame, periodo: int = 20) -> Optional[Dict]:
    try:
        bb = ta.bbands(df['Close'], length=periodo)
        if bb is not None and not bb.empty:
            upper = float(bb['BBU_20_2.0'].iloc[-1])
            mid = float(bb['BBM_20_2.0'].iloc[-1])
            lower = float(bb['BBL_20_2.0'].iloc[-1])
            close = float(df['Close'].iloc[-1])
            pos = (close - lower) / (upper - lower) * 100 if upper != lower else 50
            return {'upper': round(upper, 2), 'mid': round(mid, 2), 'lower': round(lower, 2), 'posicao_%': round(pos, 1)}
    except Exception as e:
        _log_exc('calcular_bandas_bollinger', e)
    return None

def calcular_climax_volume(df: pd.DataFrame, periodo: int = 50) -> bool:
    if len(df) < periodo:
        return False
    try:
        vol_med = df['Volume'].rolling(periodo).mean().iloc[-1]
        vol_at = df['Volume'].iloc[-1]
        return vol_at > 3 * vol_med if (pd.notna(vol_med) and vol_med > 0) else False
    except Exception:
        return False

def analisar_candle(row: pd.Series, row_ant: Optional[pd.Series] = None) -> Dict[str, bool]:
    open_, high, low, close = row['Open'], row['High'], row['Low'], row['Close']
    corpo = abs(close - open_)
    range_total = high - low
    if range_total <= 0:
        return {}
    p_sup = high - max(open_, close)
    p_inf = min(open_, close) - low
    res = {}
    if p_inf >= 2*corpo and p_sup <= 0.3*range_total and corpo > 0:
        res['martelo'] = True
    if p_sup >= 2*corpo and p_inf <= 0.3*range_total and corpo > 0:
        res['estrela_cadente'] = True
    if corpo <= 0.05 * range_total:
        res['doji'] = True
    if row_ant is not None:
        open_ant, close_ant = row_ant['Open'], row_ant['Close']
        corpo_ant = abs(close_ant - open_ant)
        if corpo < corpo_ant and high <= row_ant['High'] and low >= row_ant['Low']:
            if close_ant < open_ant and close > open_:
                res['harami_alta'] = True
            elif close_ant > open_ant and close < open_:
                res['harami_baixa'] = True
        if corpo > corpo_ant:
            if close_ant < open_ant and close > open_ and open_ <= close_ant and close >= open_ant:
                res['engolfo_alta'] = True
            if close_ant > open_ant and close < open_ and open_ >= close_ant and close <= open_ant:
                res['engolfo_baixa'] = True
        if close_ant < open_ant and close > open_ and open_ > close_ant:
            res['kicker_alta'] = True
        if close_ant > open_ant and close < open_ and open_ < close_ant:
            res['kicker_baixa'] = True
    return res

def detectar_bebe_abandonado(df: pd.DataFrame) -> Optional[Dict]:
    if len(df) < 4:
        return None
    c1, c2, c3 = df.iloc[-4], df.iloc[-3], df.iloc[-2]
    if c1['High'] <= 0 or c2['High'] <= 0 or c2['Low'] <= 0 or c3['Low'] <= 0:
        return None
    try:
        if c1['Close'] < c1['Open']:
            gap1 = (c2['Low'] - c1['High']) / c1['High']
        else:
            gap1 = (c1['Low'] - c2['High']) / c2['High']
        if c3['Close'] > c3['Open']:
            gap2 = (c3['High'] - c2['Low']) / c2['Low']
        else:
            gap2 = (c2['High'] - c3['Low']) / c3['Low']
    except (ZeroDivisionError, KeyError, TypeError):
        return None
    if gap1 > 0.02 and gap2 > 0.02 and analisar_candle(c2).get('doji'):
        if c1['Close'] < c1['Open'] and c3['Close'] > c3['Open']:
            return {'tipo': 'Bebe_abandonado_alta'}
        if c1['Close'] > c1['Open'] and c3['Close'] < c3['Open']:
            return {'tipo': 'Bebe_abandonado_baixa'}
    return None

def detectar_retangulo(df: pd.DataFrame, janela: int = 20) -> Optional[Dict]:
    if len(df) < janela:
        return None
    highs, lows = df['High'].iloc[-janela:], df['Low'].iloc[-janela:]
    resist = highs.max()
    suporte = lows.min()
    if resist - suporte < 0.02 * suporte:
        return None
    toques_resist = np.sum(highs.values >= resist * 0.99)
    toques_suporte = np.sum(lows.values <= suporte * 1.01)
    if toques_resist >= 2 and toques_suporte >= 2:
        return {'tipo': 'Retangulo', 'suporte': round(suporte, 2), 'resistencia': round(resist, 2)}
    return None

def detectar_alargamento(df: pd.DataFrame, janela: int = 20) -> Optional[Dict]:
    if len(df) < janela:
        return None
    highs = df['High'].iloc[-janela:].values
    lows = df['Low'].iloc[-janela:].values
    if highs[-1] > highs[0] and lows[-1] < lows[0]:
        return {'tipo': 'Alargamento'}
    return None

def detectar_estrutura_dow(df: pd.DataFrame) -> Optional[Dict]:
    if len(df) < 26:
        return None
    ult_top, prev_top = df['High'].iloc[-1], df['High'].iloc[-26]
    ult_fnd, prev_fnd = df['Low'].iloc[-1], df['Low'].iloc[-26]
    if ult_top > prev_top and ult_fnd > prev_fnd:
        return {'tendencia_dow': 'ALTA'}
    elif ult_top < prev_top and ult_fnd < prev_fnd:
        return {'tendencia_dow': 'BAIXA'}
    return {'tendencia_dow': 'LATERAL'}

def calcular_fibonacci_retracao(df: pd.DataFrame) -> Optional[Dict[str, float]]:
    if len(df) < 50:
        return None
    sh = detectar_swing_high(df, janela=10, confirmar=False)
    sl = detectar_swing_low(df, janela=10, confirmar=False)
    if sh is None or sl is None or sh <= sl or sl <= 0:
        return None
    diff = sh - sl
    return {'38.2%': round(sh - diff * 0.382, 2), '50.0%': round(sh - diff * 0.500, 2), '61.8%': round(sh - diff * 0.618, 2)}

def calcular_alvos_fibonacci(df: pd.DataFrame, direcao: str, fib_window: int = 20) -> Dict[str, float]:
    if len(df) < fib_window:
        return {}
    try:
        df_rec = df.iloc[-fib_window:]
        sl = detectar_swing_low(df_rec, janela=5, confirmar=False)
        sh = detectar_swing_high(df_rec, janela=5, confirmar=False)
        if sl is None or sh is None or sl <= 0 or sh <= 0 or sl >= sh:
            return {}
        amp = np.log(sh) - np.log(sl)
        base = np.log(sh)
        mults = [1.000, 1.618, 2.618, 4.236]
        if direcao == 'COMPRA':
            return {f'{m*100}%': round(np.exp(base + amp * m), 2) for m in mults}
        return {f'{m*100}%': round(np.exp(base - amp * m), 2) for m in mults}
    except Exception as e:
        _log_exc('calcular_alvos_fibonacci', e)
        return {}

# ==================== CORREÇÃO CRÍTICA: RISCO DIRECIONAL ====================
def calcular_payoff_real(entrada: float, alvo: float, stop: float, custos_pct: float) -> float:
    if stop >= entrada or alvo <= entrada:
        return 0.0
    risco = entrada - stop
    if risco <= 0:
        return 0.0
    retorno_bruto = alvo - entrada
    custo_total = (entrada + alvo) * custos_pct
    retorno_liquido = max(0, retorno_bruto - custo_total)
    return round(retorno_liquido / risco, 2)

def detectar_regime_volatilidade(serie: pd.Series, janela: int = 40) -> str:
    try:
        if serie is None or serie.empty or len(serie) < janela:
            return 'BAIXA'
        ret = serie.pct_change().dropna()
        if ret.empty or len(ret) < 20:
            return 'BAIXA'
        vol_at = ret.rolling(20).std().iloc[-1]
        vol_hist = ret.rolling(janela).std().dropna()
        if pd.isna(vol_at) or vol_hist.empty:
            return 'BAIXA'
        return 'ALTA' if (vol_hist < vol_at).mean() > 0.7 else 'BAIXA'
    except Exception as e:
        _log_exc('detectar_regime_volatilidade', e)
        return 'BAIXA'

# ==================== CORREÇÃO: EXCEPT GENÉRICO ====================
def detectar_volume_anormal(df: pd.DataFrame, periodo: int = 20, limiar: float = 1.5) -> bool:
    if len(df) < periodo:
        return False
    try:
        vol_medio = df['Volume'].rolling(periodo).mean().iloc[-1]
        vol_atual = df['Volume'].iloc[-1]
        return vol_atual >= vol_medio * limiar if (pd.notna(vol_medio) and vol_medio > 0) else False
    except Exception:
        return False

def fractional_kelly(win_rate: float, payoff_ratio: float, frac: float = 0.25) -> float:
    if payoff_ratio <= 0:
        return 0.0
    kelly = (payoff_ratio * win_rate - (1 - win_rate)) / payoff_ratio
    return max(0.0, min(kelly, 0.25)) * frac

SENTIMENTO_PADRAO = 0.0
print("✅ Célula 2 carregada.")

In [ ]:
# =============================================================================
# CÉLULA 3: GUARDIÕES E ANÁLISE (v8.5.3 - FINAL)
# =============================================================================

SETOR_POR_TICKER = {
    'PETR4': 'Petróleo', 'PETR3': 'Petróleo', 'PRIO3': 'Petróleo',
    'VALE3': 'Mineração', 'GGBR4': 'Siderurgia', 'CSNA3': 'Siderurgia',
    'ITUB4': 'Financeiro', 'BBDC4': 'Financeiro', 'BBAS3': 'Financeiro',
    'ABEV3': 'Consumo', 'MGLU3': 'Varejo', 'RENT3': 'Varejo',
    'WEGE3': 'Indústria', 'RADL3': 'Saúde', 'JBSS3': 'Alimentos',
}

def obter_setor(ticker: str) -> str:
    t_clean = ticker.replace('.SA', '')
    return SETOR_POR_TICKER.get(t_clean, 'Outros')

MACRO_REFERENCE = {
    'VALE3.SA': ('GC=F', 0.6),
    'PETR4.SA': ('CL=F', 0.8),
    'PETR3.SA': ('CL=F', 0.8),
    'ABEV3.SA': ('CORN', 0.3)
}

_cache_macro_data, _cache_macro_timestamp = {}, {}

def _obter_dados_macro(bench: str):
    agora = datetime.now()
    if bench in _cache_macro_timestamp:
        if (agora - _cache_macro_timestamp[bench]).total_seconds() / 3600 < CACHE_MACRO_EXPIRY_HORAS:
            return _cache_macro_data.get(bench)
    try:
        df_b = yf.download(bench, period='1y', interval='1wk', progress=False, auto_adjust=True)
        vals = df_b['Close'].values if not df_b.empty else None
        _cache_macro_data[bench] = vals
        _cache_macro_timestamp[bench] = agora
        return vals
    except Exception as e:
        _log_exc('_obter_dados_macro', e)
        return None

# ==================== CORREÇÃO: WARM-UP EMA 150 E SCORE NEUTRO ====================
def verificar_alinhamento_macro(ticker: str, direcao: str, _=None) -> Tuple[bool, int]:
    if ticker not in MACRO_REFERENCE:
        return True, 0  # Score neutro para ativos sem macro
    ref, _ = MACRO_REFERENCE[ticker]
    precos = _obter_dados_macro(ref)
    if precos is None or len(precos) < 150:  # warm-up de 150 períodos
        return True, 0
    ema50 = pd.Series(precos).ewm(span=50, adjust=False).mean()
    e_at, e_lag = ema50.iloc[-1], ema50.iloc[-5]
    if pd.isna(e_at) or pd.isna(e_lag):
        return True, 0
    slope_pos = e_at > e_lag
    if direcao == 'COMPRA' and precos[-1] > e_at and slope_pos:
        return True, 30
    if direcao == 'VENDA' and precos[-1] < e_at and not slope_pos:
        return True, 30
    return False, 0

def validar_toque_zona_wyckoff(df, lta, banda_pct=0.01, volume_min_ratio=0.8):
    if len(df) < 2 or lta is None or lta <= 0:
        return False, 'indefinido'
    ult = df.iloc[-1]
    low, close = ult['Low'], ult['Close']
    vol_atual = ult['Volume']
    vol_med = df['Volume'].rolling(20).mean().iloc[-1] if len(df) >= 20 else vol_atual
    zona_inf = lta * (1 - banda_pct)
    zona_sup = lta * (1 + banda_pct)
    toque_valido = (low <= zona_sup) and (close >= zona_inf)
    if not toque_valido:
        return False, 'indefinido'
    suporte_range = df['Low'].rolling(20).min().iloc[-1]
    resistencia_range = df['High'].rolling(20).max().iloc[-1]
    range_total = resistencia_range - suporte_range
    posicao_no_range = (close - suporte_range) / range_total if range_total > 0 else 0.5
    if posicao_no_range < 0.4:
        regime = 'acumulacao'
        toque_valido = vol_atual >= vol_med * volume_min_ratio if (pd.notna(vol_med) and vol_med > 0) else True
    elif posicao_no_range > 0.6:
        regime = 'markup'
        toque_valido = vol_atual >= vol_med * 1.2 if (pd.notna(vol_med) and vol_med > 0) else True
    else:
        regime = 'neutro'
        toque_valido = vol_atual >= vol_med * 0.8 if (pd.notna(vol_med) and vol_med > 0) else True
    return toque_valido, regime

def detectar_fase_wyckoff_adaptativo(df_w, suporte=None, resistencia=None, atr_period=14):
    if len(df_w) < 30:
        return 'INDEFINIDO', 0.0
    try:
        atr = ta.atr(df_w['High'], df_w['Low'], df_w['Close'], length=atr_period)
        if atr is None or atr.empty:
            return 'INDEFINIDO', 0.0
        atr_med = atr.rolling(20).mean()
        atr_val, atr_med_val = atr.iloc[-1], atr_med.iloc[-1]
        vol_rel = atr_val / atr_med_val if (pd.notna(atr_med_val) and atr_med_val > 0) else 1.0
    except Exception:
        return 'INDEFINIDO', 0.0
    thr_base = PARAMS_ATIVOS.get('wyckoff_threshold', 0.75)
    thr_comp = max(0.65, min(0.90, thr_base + 0.10 * (vol_rel - 1)))
    range_s = df_w['High'] - df_w['Low']
    r_med = range_s.rolling(20).mean().iloc[-1]
    if range_s.iloc[-1] >= r_med * thr_comp:
        return 'INDEFINIDO', 0.0
    px = df_w['Close'].iloc[-1]
    suporte = suporte if suporte else df_w['Low'].rolling(20).min().iloc[-1]
    resistencia = resistencia if resistencia else df_w['High'].rolling(20).max().iloc[-1]
    faixa = resistencia - suporte
    if faixa <= 0:
        return 'INDEFINIDO', 0.0
    pos_rel = (px - suporte) / faixa
    candles = df_w.iloc[-8:]
    alta = candles['Close'] > candles['Open']
    baixa = candles['Close'] < candles['Open']
    vol_alta = candles.loc[alta, 'Volume'].mean() if alta.any() else 0
    vol_baixa = candles.loc[baixa, 'Volume'].mean() if baixa.any() else 0
    if vol_baixa == 0:
        return 'INDEFINIDO', 0.0
    razao = vol_alta / vol_baixa
    conf = min(1.0, (r_med - range_s.iloc[-1]) / r_med) if r_med > 0 else 0.5
    if pos_rel < 0.4 and razao > 1.2:
        return 'ACUMULACAO', conf
    if pos_rel > 0.6 and razao < 0.8:
        return 'DISTRIBUICAO', conf
    return 'INDEFINIDO', 0.0

# ==================== CORREÇÃO: NORMALIZAÇÃO SEGURA ====================
def _normalizar_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join(col).strip() for col in df.columns.values]
    rename_map = {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}
    lowercase = {c: c.lower() for c in df.columns}
    df.rename(columns={col: rename_map.get(lowercase.get(col, col), col) for col in df.columns}, inplace=True)
    df.sort_index(inplace=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def _safe_atr(df_high, df_low, df_close, length: int) -> float:
    try:
        atr_series = ta.atr(df_high, df_low, df_close, length=length)
        if atr_series is None or atr_series.empty:
            return 0.0
        val = atr_series.iloc[-1]
        return float(val) if pd.notna(val) else 0.0
    except Exception as e:
        _log_exc('_safe_atr', e)
        return 0.0

# Guardiões (mantidos, apenas chamam as funções corrigidas)
def guardiao_dow(df_w, contexto_trap, modo_gebra) -> Tuple[bool, Optional[str]]:
    if modo_gebra != 'black_belt':
        return True, None
    dow = detectar_estrutura_dow(df_w)
    ok = contexto_trap or (dow and dow['tendencia_dow'] == 'ALTA')
    return ok, 'Dow' if not ok else None

def guardiao_elliott(df_w, contexto_trap, modo_gebra) -> Tuple[bool, Optional[str], bool]:
    if modo_gebra != 'black_belt':
        return True, None, False
    swing_highs, swing_lows = [], []
    for j in range(5, len(df_w)-5):
        if df_w['High'].values[j] >= max(df_w['High'].values[j-5:j+6]):
            swing_highs.append(df_w['High'].values[j])
        if df_w['Low'].values[j] <= min(df_w['Low'].values[j-5:j+6]):
            swing_lows.append(df_w['Low'].values[j])
    valido, _ = validar_elliott(df_w, swing_lows, swing_highs)
    ok = contexto_trap or valido
    return ok, 'Elliott' if not ok else None, valido

def guardiao_fibonacci(df_w, entrada) -> Tuple[bool, Optional[str]]:
    fib_ret = calcular_fibonacci_retracao(df_w)
    if fib_ret:
        ok = (fib_ret['61.8%'] <= entrada <= fib_ret['38.2%'])
        return ok, 'Fibonacci' if not ok else None
    return True, None

def guardiao_retangulo(df_w, entrada, modo_gebra) -> Tuple[bool, Optional[str]]:
    if modo_gebra != 'black_belt':
        return True, None
    ret = detectar_retangulo(df_w)
    if ret and 'suporte' in ret:
        ok = entrada <= ret['suporte'] * 1.05
        return ok, 'Retângulo' if not ok else None
    return True, None

def guardiao_estocastico(df_w, modo_gebra) -> Tuple[bool, Optional[str]]:
    if modo_gebra != 'black_belt':
        return True, None
    stoch_k, _ = calcular_estocastico(df_w)
    ok = stoch_k is not None and stoch_k < 30
    return ok, 'Estocástico' if not ok else None

def guardiao_medias(df_w, entrada, modo_gebra) -> Tuple[bool, Optional[str]]:
    if modo_gebra != 'black_belt':
        return True, None
    mm200w = df_w['Close'].rolling(200).mean().iloc[-1]
    mm200w_ant = df_w['Close'].rolling(200).mean().iloc[-5] if len(df_w) >= 200 else mm200w
    ok = pd.notna(mm200w) and entrada > mm200w and mm200w > mm200w_ant
    return ok, 'Médias' if not ok else None

def guardiao_zona_wyckoff(df_w, lta, banda_pct) -> Tuple[bool, Optional[str], str]:
    toca, regime = validar_toque_zona_wyckoff(df_w, lta, banda_pct)
    return toca, 'Zona Wyckoff' if not toca else None, regime

def guardiao_gatilho(df_w) -> Tuple[bool, Optional[str], Dict]:
    pc = analisar_candle(df_w.iloc[-1], df_w.iloc[-2] if len(df_w) >= 2 else None)
    ok = pc.get('martelo') or pc.get('engolfo_alta') or pc.get('kicker_alta') or pc.get('harami_alta')
    return ok, 'Gatilho' if not ok else None, pc

def guardiao_payoff(entrada, alvo, stop, custos_pct, minimo=3.0) -> Tuple[bool, Optional[str], float]:
    p_real = calcular_payoff_real(entrada, alvo, stop, custos_pct)
    ok = p_real >= minimo
    return ok, 'Payoff' if not ok else None, p_real

def guardiao_corda(df_w, entrada, modo_gebra, dist_max=30.0) -> Tuple[bool, Optional[str]]:
    if modo_gebra != 'black_belt':
        return True, None
    mm200w_val = df_w['Close'].rolling(200).mean().iloc[-1]
    if pd.notna(mm200w_val) and mm200w_val > 0:
        dist = (entrada - mm200w_val) / mm200w_val * 100
        ok = dist <= dist_max
        return ok, 'Corda' if not ok else None
    return True, None

def guardiao_macd(df_w, usar_macd) -> Tuple[bool, Optional[str]]:
    if not usar_macd:
        return True, None
    _, _, macd_hist = calcular_macd(df_w)
    ok = macd_hist is not None and macd_hist > 0
    return ok, 'MACD' if not ok else None

def guardiao_setor(ticker, contagem_setores, max_por_setor) -> Tuple[bool, Optional[str], str]:
    setor = obter_setor(ticker)
    ok = contagem_setores.get(setor, 0) < max_por_setor
    return ok, f'Setor ({setor})' if not ok else None, setor

def calcular_score_qualidade(r, dow_ok, elliott_valido, fib_ok, ret_ok, stoch_ok, ma_ok, toca_zona, gatilho_ok, payoff_ok, corda_ok, alta_conf_ok, bollinger_ok, contexto_trap, is_arm, eficiencia, regime_wyckoff) -> int:
    score = 50
    if contexto_trap:
        score += 20
    if is_arm:
        score += 10
    if eficiencia and eficiencia > 0.8:
        score += 15
    elif eficiencia and eficiencia > 0.6:
        score += 8
    if elliott_valido:
        score += 10
    if dow_ok and not contexto_trap:
        score += 5
    if regime_wyckoff == 'acumulacao':
        score += 5
    elif regime_wyckoff == 'markup':
        score += 3
    if payoff_ok and r.get('Payoff Real', 0) > 4.0:
        score += 5
    return min(100, max(0, score))

logger.log("✅ Guardiões v8.5.3 carregados", "INFO")
print("✅ Célula 3 carregada.")

In [ ]:
# =============================================================================
# CÉLULA 4: EXECUÇÃO PRINCIPAL (v8.5.3 – FINAL)
# =============================================================================

try:
    from google.colab import userdata
    if not EMAIL_REMETENTE:
        EMAIL_REMETENTE = userdata.get('TRADING_EMAIL')
    if not SENHA_APP:
        SENHA_APP = userdata.get('GMAIL_APP_PASSWORD')
except Exception as e:
    logger.warn(f"userdata indisponível (normal fora do Colab): {e}")

def montar_tabela_html(oportunidades, titulo, regime_vol, nh_nl=None, lad=None):
    if not oportunidades:
        return ""
    corpo = f"<h3>{titulo}</h3><table border='1' cellpadding='3' cellspacing='0' style='border-collapse:collapse;font-size:12px'>"
    corpo += "<tr><th>Ticker</th><th>Dir.</th><th>Setor</th><th>Entrada</th><th>Stop</th><th>Alvo</th><th>Payoff</th><th>Score</th><th>Padrões</th><th>Lote</th></tr>"
    for op in oportunidades:
        setor = obter_setor(op['Ticker'])
        padroes = op.get('Padrões Detectados', '—')
        score = op.get('Score Qualidade', '—')
        corpo += f"<tr><td>{op['Ticker']}</td><td>{op['Direcao']}</td><td>{setor}</td><td>R$ {op['Entrada']:.2f}</td><td>R$ {op['Stop Loss']:.2f}</td><td>R$ {op.get('Alvo Recomendado', 0):.2f}</td><td>{op.get('Payoff Real', 0)}:1</td><td>{score}</td><td>{padroes}</td><td>{op.get('Lote', 0)}</td></tr>"
    corpo += f"</table><br><small>Custos: {PARAMS_ATIVOS['custos_pct']*100:.1f}% | Regime: {regime_vol} | NH‑NL: {nh_nl}% | LAD: {lad['saldo'] if lad else 'N/A'} | Modo: {MODO_GEBRA} | v8.5.3</small>"
    return corpo

def gerar_relatorio_completo(logger, stats_filtros, status_ativos, tickers_liquidos, regime_vol, nh_nl, lad, oportunidades_swing, oportunidades_position, contagem_setores, kelly_pct, tickers_processados) -> str:
    relatorio = []
    relatorio.append("=" * 80)
    relatorio.append(f"RELATÓRIO COMPLETO DO SISTEMA DE TRADING - v8.5.3")
    relatorio.append(f"Data/Hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    relatorio.append("=" * 80 + "\n")
    relatorio.append("📌 1. PARÂMETROS DO SISTEMA")
    relatorio.append("-" * 50)
    relatorio.append(f"Capital Total: R$ {CAPITAL_TOTAL:,.2f}")
    relatorio.append(f"Modo GEBRA: {MODO_GEBRA}")
    relatorio.append(f"Kelly: {PARAMS_ATIVOS.get('kelly_frac', 0.25)}")
    relatorio.append(f"Custos: {PARAMS_ATIVOS.get('custos_pct', 0.005)*100:.2f}%\n")
    relatorio.append("📊 2. INFORMAÇÕES DE MERCADO")
    relatorio.append("-" * 50)
    relatorio.append(f"Regime de Volatilidade: {regime_vol}")
    relatorio.append(f"NH-NL: {nh_nl}%")
    if lad:
        relatorio.append(f"LAD: Avanços={lad.get('avancos', 0)} Declínios={lad.get('declinios', 0)} Saldo={lad.get('saldo', 0)}")
    relatorio.append(f"\n📈 3. ANÁLISE ({len(tickers_processados)} ativos)")
    relatorio.append("-" * 50)
    relatorio.append(f"Líquidos: {len(tickers_liquidos)}")
    relatorio.append(f"Analisados: {stats_filtros['total_analisados']}")
    relatorio.append(f"Aprovados Swing: {stats_filtros['setup_aprovado_swing']}")
    relatorio.append(f"Aprovados Position: {stats_filtros['setup_aprovado_position']}")
    relatorio.append(f"\n🔍 4. FILTROS APLICADOS")
    relatorio.append("-" * 50)
    for key, count in stats_filtros.items():
        if count > 0 and key.startswith('bloqueios_'):
            relatorio.append(f"  • {key.replace('bloqueios_', '').replace('_', ' ').title()}: {count}")
    if status_ativos:
        recusados = [s for s in status_ativos if 'Recusado' in str(s.get('Status', ''))]
        aprovados = [s for s in status_ativos if 'APROVADO' in str(s.get('Status', ''))]
        relatorio.append(f"\n📋 5. STATUS ({len(aprovados)} aprovados, {len(recusados)} recusados)")
        if aprovados:
            relatorio.append("✅ APROVADOS:")
            for s in aprovados[:20]:
                relatorio.append(f"  • {s['Ticker']}")
        if recusados:
            motivos = Counter([s.get('Filtro', 'Desconhecido') for s in recusados])
            for motivo, qtd in motivos.most_common(10):
                relatorio.append(f"  • {motivo}: {qtd} ativos")
    relatorio.append(f"\n🎯 6. OPORTUNIDADES")
    relatorio.append("-" * 50)
    relatorio.append(f"Kelly %: {kelly_pct*100:.2f}%")
    if oportunidades_swing:
        relatorio.append(f"\nSWING ({len(oportunidades_swing)}):")
        for i, op in enumerate(oportunidades_swing[:10], 1):
            relatorio.append(f"  {i}. {op['Ticker']} | {op['Direcao']} | E: R${op['Entrada']:.2f} | Score: {op.get('Score Qualidade', 0)}")
    else:
        relatorio.append("\nSWING: Nenhuma oportunidade")
    if hasattr(logger, 'timings') and logger.timings:
        total_time = sum(v.get('duracao', 0) for v in logger.timings.values() if 'duracao' in v)
        relatorio.append(f"\n⏱️ 7. PERFORMANCE: {total_time:.2f}s")
        for etapa, dados in logger.timings.items():
            if 'duracao' in dados:
                relatorio.append(f"  • {etapa}: {dados['duracao']:.2f}s")
    relatorio.append("\n" + "=" * 80)
    relatorio.append(f"FIM DO RELATÓRIO - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    relatorio.append("=" * 80)
    return "\n".join(relatorio)

def enviar_log_sempre(logger, stats_filtros, status_ativos, tickers_liquidos, regime_vol, nh_nl, lad, oportunidades_swing, oportunidades_position, contagem_setores, kelly_pct, data_d, data_w, tickers_processados) -> None:
    relatorio_texto = gerar_relatorio_completo(logger, stats_filtros, status_ativos, tickers_liquidos, regime_vol, nh_nl, lad, oportunidades_swing, oportunidades_position, contagem_setores, kelly_pct, tickers_processados)
    nome_arquivo = f"trading_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    with open(nome_arquivo, 'w', encoding='utf-8') as f:
        f.write(relatorio_texto)
    logger.log(f"📁 Relatório salvo: {nome_arquivo}", "INFO")
    if EMAIL_REMETENTE and SENHA_APP:
        try:
            msg = MIMEMultipart()
            msg['From'] = EMAIL_REMETENTE
            msg['To'] = EMAIL_REMETENTE
            assunto = f"🚀 SISTEMA DE TRADING - {len(oportunidades_swing)} Ops - {datetime.now().strftime('%d/%m/%Y %H:%M')}" if oportunidades_swing else f"📊 SISTEMA DE TRADING - SEM OPORTUNIDADES - {datetime.now().strftime('%d/%m/%Y %H:%M')}"
            msg['Subject'] = assunto
            corpo_html = f"<html><body style='font-family:monospace;white-space:pre-wrap'><h2>📈 Sistema de Trading GEBRA - Relatório v8.5.3</h2><pre>{relatorio_texto}</pre></body></html>"
            msg.attach(MIMEText(corpo_html, 'html'))
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
                server.login(EMAIL_REMETENTE, SENHA_APP)
                server.send_message(msg)
            logger.log(f"📧 Email enviado: {assunto}", "INFO")
        except Exception as e:
            logger.log(f"⚠️ Falha email: {str(e)[:200]}", "WARN")
    else:
        logger.log(f"⚠️ Email não configurado. Relatório: {nome_arquivo}", "WARN")
    try:
        from google.colab import files
        files.download(nome_arquivo)
        logger.log(f"📥 Download iniciado: {nome_arquivo}", "INFO")
    except:
        pass

def obter_tickers_b3() -> List[str]:
    if os.path.exists(CACHE_TICKERS_FILE):
        try:
            with open(CACHE_TICKERS_FILE, 'r', encoding='utf-8') as f:
                cache = json.load(f)
            if (datetime.now() - datetime.fromisoformat(cache['timestamp'])).total_seconds() / 3600 < 24:
                logger.log(f"📦 Cache tickers ({len(cache['tickers'])} ativos)", "INFO")
                return cache['tickers']
        except:
            pass
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        response = requests.get("https://www.dadosdemercado.com.br/acoes", timeout=10, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        tickers = []
        for row in soup.select('table tbody tr'):
            cells = row.find_all('td')
            if cells and not cells[0].text.strip().startswith('#'):
                ticker = cells[0].text.strip().replace('.SA', '')
                if ticker:
                    tickers.append(ticker)
        if tickers:
            with open(CACHE_TICKERS_FILE, 'w', encoding='utf-8') as f:
                json.dump({'timestamp': datetime.now().isoformat(), 'tickers': tickers}, f)
            logger.log(f"🌐 Scraping ({len(tickers)} ativos)", "INFO")
            return tickers
    except Exception as e:
        logger.log(f"⚠️ Scraping falhou: {str(e)[:80]}", "WARN")
    logger.log("🔄 Fallback tickers", "WARN")
    return FALLBACK_TICKERS.copy()

# ==================== CORREÇÃO: LOOK-AHEAD BIAS (CLOSED/RIGHT) ====================
def resample_tf(df: pd.DataFrame, freq: str, min_days: int = 4, min_days_monthly: int = 10) -> Optional[pd.DataFrame]:
    if df is None or df.empty:
        return None
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            return None
    hoje = datetime.now()
    dia_semana = hoje.weekday()
    hora_atual = hoje.hour
    if freq.startswith('W'):
        if dia_semana < 4 or (dia_semana == 4 and hora_atual < 18):
            ultima_sexta = df.index[df.index.dayofweek == 4]
            if len(ultima_sexta) > 0:
                df = df.loc[:ultima_sexta[-1]]
                if df.empty:
                    return None
    agg = {'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 'Volume': 'sum'}
    # CORREÇÃO: closed='right', label='right' elimina look-ahead
    df_r = df.resample(freq, closed='right', label='right').agg(agg)
    if freq.startswith('W'):
        counts = df.resample(freq, closed='right', label='right').count()['Close']
        df_r = df_r[counts >= min_days]
    elif freq in ('ME', 'M'):
        counts = df.resample(freq, closed='right', label='right').count()['Close']
        df_r = df_r[counts >= min_days_monthly]
    df_r = df_r.replace([np.inf, -np.inf], np.nan).dropna()
    df_r = df_r[df_r['Close'] > 0]
    return df_r

# ==================== CORREÇÃO: EXTRAÇÃO SEGURA DO MULTIINDEX ====================
def extrair_dataframe_ticker(data_raw, ticker: str) -> Optional[pd.DataFrame]:
    try:
        if isinstance(data_raw.columns, pd.MultiIndex):
            if (ticker not in data_raw.columns.get_level_values(1) and 
                ticker not in data_raw.columns.get_level_values(0)):
                return None
            df = data_raw[ticker].copy()
        else:
            df = data_raw.copy()
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = ['_'.join(col).strip() for col in df.columns.values]
        df.columns = [c.lower() for c in df.columns]
        rename_map = {'close':'Close','high':'High','low':'Low','open':'Open','volume':'Volume'}
        df.rename(columns={k: rename_map.get(k, k) for k in df.columns if k in rename_map}, inplace=True)
        return df
    except Exception as e:
        _log_exc(f'extrair_dataframe_ticker {ticker}', e)
        return None

def verificar_circuit_breakers() -> Tuple[bool, Optional[str]]:
    if not os.path.exists(ARQUIVO_LOG):
        return True, None
    try:
        with open(ARQUIVO_LOG, 'r', encoding='utf-8') as f:
            logs = json.load(f)
        if not isinstance(logs, list):
            return False, "Falha de segurança: Log corrompido (formato inválido)"
        hoje = datetime.now().date()
        trades = []
        for l in logs:
            try:
                if l.get('tipo') == 'TRADE_FECHADO':
                    ts = datetime.fromisoformat(l['timestamp'])
                    if ts.date() == hoje:
                        trades.append(l)
            except (KeyError, ValueError):
                continue
        if not trades:
            return True, None
        pnl = 0
        for t in trades:
            try:
                pnl += float(t['dados'].get('pnl_real', 0))
            except:
                continue
        if abs(pnl) / CAPITAL_TOTAL >= DRAWDOWN_MAX_DIARIO:
            return False, f"Drawdown >= {DRAWDOWN_MAX_DIARIO*100:.1f}%"
        perdas = 0
        for t in sorted(trades, key=lambda x: x.get('timestamp', ''), reverse=True):
            try:
                if float(t['dados'].get('pnl_real', 0)) < 0:
                    perdas += 1
                else:
                    break
            except:
                break
        if perdas >= MAX_PERDAS_CONSECUTIVAS:
            return False, f"{perdas} perdas consecutivas"
        return True, None
    except json.JSONDecodeError:
        return False, "Falha de segurança: Log de risco corrompido (JSON inválido)"
    except Exception as e:
        logger.error(f"Circuit Breaker: Falha crítica - {str(e)[:200]}")
        return False, f"Falha de segurança ao verificar logs: {str(e)[:100]}"

# ========== EXECUÇÃO PRINCIPAL ==========
tickers_processados = []

logger.iniciar_etapa("Coleta de Tickers")
tickers_b3 = obter_tickers_b3()
tickers_b3 = [t.replace('.SA', '') for t in tickers_b3]
logger.concluir_etapa("Coleta de Tickers", {'total': len(tickers_b3)})

tickers_yahoo = [t + ".SA" for t in tickers_b3]
tickers_liquidos = []
BATCH = 50

logger.iniciar_etapa("Filtro de Liquidez")
for i in range(0, len(tickers_yahoo), BATCH):
    batch = tickers_yahoo[i:i+BATCH]
    try:
        data_raw = yf.download(batch, period='3mo', interval='1d', group_by='ticker', progress=False, auto_adjust=True)
        for t in batch:
            if t in TICKERS_BLOQUEADOS:
                continue
            df = extrair_dataframe_ticker(data_raw, t)
            if df is None or df.empty or 'Volume' not in df.columns:
                continue
            try:
                vol_med = df['Volume'].rolling(21).mean().iloc[-1]
                preco = df['Close'].iloc[-1]
                if pd.isna(vol_med) or pd.isna(preco) or preco <= 0:
                    continue
                if (vol_med >= VOLUME_MINIMO_ACAO and (vol_med * preco) >= VOLUME_FINANCEIRO_MINIMO):
                    tickers_liquidos.append(t)
            except Exception as e:
                _log_exc(f'Filtro liquidez {t}', e)
                continue
    except Exception as e:
        logger.log(f"Erro baixando lote {i//BATCH}: {str(e)[:100]}", "ERRO")
    time.sleep(1)

if len(tickers_liquidos) < 10:
    logger.log("Poucos ativos, usando fallback", "WARN")
    tickers_liquidos = [t + ".SA" for t in FALLBACK_TICKERS[:20]]

logger.concluir_etapa("Filtro de Liquidez", {'liquidos': len(tickers_liquidos)})

logger.iniciar_etapa("Download Dados (5 anos)")
data_d = {}
falhas = []
for i in range(0, len(tickers_liquidos), BATCH):
    batch = tickers_liquidos[i:i+BATCH]
    try:
        data_raw = yf.download(batch, period='5y', interval='1d', group_by='ticker', progress=False, auto_adjust=True)
        for t in batch:
            df = extrair_dataframe_ticker(data_raw, t)
            if df is not None and not df.empty:
                data_d[t] = df
            else:
                falhas.append(t)
    except Exception as e:
        logger.log(f"Erro download 5y lote {i//BATCH}: {str(e)[:100]}", "ERRO")
        falhas.extend(batch)
    time.sleep(1)

logger.concluir_etapa("Download Dados", {'sucesso': len(data_d), 'falhas': len(falhas)})

logger.iniciar_etapa("Resample Semanal/Mensal")
data_w, data_m = {}, {}
for t in tickers_liquidos:
    try:
        if t in data_d and not data_d[t].empty:
            df_d = data_d[t].copy()
            data_w[t] = resample_tf(df_d, 'W-FRI')
            data_m[t] = resample_tf(df_d, 'ME', min_days_monthly=10)
    except Exception as e:
        _log_exc(f'Resample {t}', e)
        continue

logger.concluir_etapa("Resample", {'semanais': len(data_w), 'mensais': len(data_m)})

def calcular_nh_nl_simplificado(tickers_list, data_w_dict):
    count, total = 0, 0
    for t in tickers_list:
        if t not in data_w_dict or data_w_dict[t] is None or data_w_dict[t].empty:
            continue
        try:
            mm50 = data_w_dict[t]['Close'].rolling(50).mean().iloc[-1]
            close = data_w_dict[t]['Close'].iloc[-1]
            if pd.notna(mm50) and close > mm50:
                count += 1
            total += 1
        except Exception:
            continue
    return round(count / total * 100, 1) if total > 0 else None

def calcular_lad(tickers_list, data_d_dict):
    avancos, declinios, total = 0, 0, 0
    for t in tickers_list:
        if t not in data_d_dict or data_d_dict[t] is None or data_d_dict[t].empty:
            continue
        try:
            close = data_d_dict[t]['Close'].iloc[-1]
            close_ant = data_d_dict[t]['Close'].iloc[-2]
            if close > close_ant:
                avancos += 1
            elif close < close_ant:
                declinios += 1
            total += 1
        except Exception:
            continue
    return {'avancos': avancos, 'declinios': declinios, 'total': total, 'saldo': avancos - declinios}

nh_nl = calcular_nh_nl_simplificado(tickers_liquidos, data_w)
lad = calcular_lad(tickers_liquidos, data_d)
logger.log(f"📊 NH‑NL: {nh_nl}% | LAD saldo: {lad['saldo'] if lad else 'N/A'}", "INFO")

logger.iniciar_etapa("Regime de Volatilidade")
ibov = None
for simbolo in ["^BVSP", "^IBOV", "BOVA11.SA"]:
    try:
        ibov_raw = yf.download(simbolo, period='3mo', interval='1d', progress=False)
        if not ibov_raw.empty and 'Close' in ibov_raw.columns:
            ibov = ibov_raw['Close'].dropna()
            if len(ibov) >= 60:
                break
    except Exception:
        pass
if ibov is not None and len(ibov) >= 60:
    regime_vol = detectar_regime_volatilidade(ibov)
else:
    regime_vol = 'BAIXA'
PARAMS_ATIVOS.clear()
PARAMS_ATIVOS.update(PARAMS_ALTA_VOL if regime_vol == 'ALTA' else PARAMS_BAIXA_VOL)
logger.concluir_etapa("Regime", {'regime': regime_vol})

kelly_pct = fractional_kelly(WIN_RATE_ESTIMADO, PAYOFF_ESTIMADO, PARAMS_ATIVOS['kelly_frac'])
risco_maximo = CAPITAL_TOTAL * kelly_pct

pode, motivo = verificar_circuit_breakers()
if not pode:
    logger.log(f"🛑 CIRCUIT BREAKER ATIVADO: {motivo}", "ALERT")
    enviar_log_sempre(logger, {'total_analisados': 0}, [], tickers_liquidos, regime_vol, nh_nl, lad, [], [], {}, kelly_pct, data_d, data_w, tickers_liquidos)
    raise SystemExit("Circuit Breaker ativado")

oportunidades_swing, oportunidades_position = [], []
status_ativos = []
contagem_setores = {}
tickers_processados = tickers_liquidos.copy()
stats_filtros = {key:0 for key in ['total_analisados','setup_aprovado_swing','setup_aprovado_position','bloqueios_preco','bloqueios_risco','bloqueios_confluencia','bloqueios_mm200','bloqueios_volume','bloqueios_lta','bloqueios_dow','bloqueios_elliott','bloqueios_fib','bloqueios_ret','bloqueios_stoch','bloqueios_ma','bloqueios_zona_wyckoff','bloqueios_gatilho','bloqueios_payoff','bloqueios_corda','bloqueios_macd','bloqueios_setor']}

preco_minimo = PARAMS_ATIVOS.get('preco_minimo', PRECO_MINIMO)
risco_max_pct = PARAMS_ATIVOS.get('risco_percentual_maximo', RISCO_PERCENTUAL_MAXIMO)

logger.iniciar_etapa("Análise de Setups")

for i, ticker in enumerate(tickers_liquidos):
    if LOG_FILTROS_DETALHADO and i % 20 == 0:
        logger.log(f"Progresso: {i+1}/{len(tickers_liquidos)}", "DEBUG")
    
    df_w = data_w.get(ticker)
    if df_w is None or df_w.empty:
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'Sem dados semanais'})
        continue
    
    stats_filtros['total_analisados'] += 1
    vol_fin = None
    try:
        vfc = (df_w['Volume'] * df_w['Close']).rolling(20).mean()
        vol_fin = vfc.iloc[-1] if pd.notna(vfc.iloc[-1]) else None
    except Exception:
        pass
    
    df_w_norm = _normalizar_dataframe(df_w)
    df_w_norm['Eficiencia'] = calcular_eficiencia_candle(df_w_norm)
    df_w_norm['Regime'] = detectar_regime(df_w_norm)
    ult = df_w_norm.iloc[-1]
    entrada = float(ult['Close'])
    if pd.isna(entrada) or entrada <= 0 or entrada < preco_minimo:
        stats_filtros['bloqueios_preco'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'Preço inválido'})
        continue
    
    rh = float(df_w_norm['High'].rolling(window=min(52, len(df_w_norm))).max().iloc[-1])
    rl = float(df_w_norm['Low'].rolling(window=min(52, len(df_w_norm))).min().iloc[-1])
    reg = int(ult['Regime']) if not pd.isna(ult['Regime']) else -1
    ef = round(float(ult['Eficiencia']), 2) if not pd.isna(ult['Eficiencia']) else None
    
    if reg not in [1, 2] or ef is None or ef < 0.6:
        stats_filtros['bloqueios_confluencia'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'Confluência'})
        continue
    
    mm200w = df_w_norm['Close'].rolling(200).mean().iloc[-1]
    if pd.notna(mm200w) and entrada < mm200w:
        stats_filtros['bloqueios_mm200'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'MM200'})
        continue
    
    if PARAMS_ATIVOS.get('exigir_volume_anormal', False) and not detectar_volume_anormal(df_w_norm):
        stats_filtros['bloqueios_volume'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'Volume'})
        continue
    
    res_lta = calcular_lta_adaptativo(df_w_norm)
    if res_lta is None:
        stats_filtros['bloqueios_lta'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'LTA'})
        continue
    lta_val = res_lta[0]
    alargamento_detectado = detectar_alargamento(df_w_norm)
    is_arm_baixa, _ = detectar_armadilha_lta(df_w_norm, rl, BANDA_ZONA_PCT)
    contexto_trap = is_arm_baixa and (alargamento_detectado is not None)
    
    atr = _safe_atr(df_w_norm['High'], df_w_norm['Low'], df_w_norm['Close'], 14) or entrada * 0.02
    stop_atr = entrada - 1.8 * atr
    swing_low_val = detectar_swing_low(df_w_norm, janela=12)
    stop_candidatos = [s for s in [stop_atr, swing_low_val] if s is not None and s > 0 and s < entrada]
    if not stop_candidatos:
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'Stop inválido'})
        continue
    stop_final = max(stop_candidatos)
    risco = entrada - stop_final
    if risco / entrada < RISCO_PERCENTUAL_MINIMO or risco / entrada > risco_max_pct:
        stats_filtros['bloqueios_risco'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': 'Risco fora'})
        continue
    alvo = entrada + risco * 3
    
    # Guardiões
    ok_dow, lbl_dow = guardiao_dow(df_w_norm, contexto_trap, MODO_GEBRA)
    if not ok_dow:
        stats_filtros['bloqueios_dow'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_dow})
        continue
    ok_ell, lbl_ell, ell_valido = guardiao_elliott(df_w_norm, contexto_trap, MODO_GEBRA)
    if not ok_ell:
        stats_filtros['bloqueios_elliott'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_ell})
        continue
    ok_fib, lbl_fib = guardiao_fibonacci(df_w_norm, entrada)
    if not ok_fib:
        stats_filtros['bloqueios_fib'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_fib})
        continue
    ok_ret, lbl_ret = guardiao_retangulo(df_w_norm, entrada, MODO_GEBRA)
    if not ok_ret:
        stats_filtros['bloqueios_ret'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_ret})
        continue
    ok_stoch, lbl_stoch = guardiao_estocastico(df_w_norm, MODO_GEBRA)
    if not ok_stoch:
        stats_filtros['bloqueios_stoch'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_stoch})
        continue
    ok_ma, lbl_ma = guardiao_medias(df_w_norm, entrada, MODO_GEBRA)
    if not ok_ma:
        stats_filtros['bloqueios_ma'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_ma})
        continue
    ok_zona, lbl_zona, regime_wyckoff = guardiao_zona_wyckoff(df_w_norm, lta_val, BANDA_ZONA_PCT)
    if not ok_zona:
        stats_filtros['bloqueios_zona_wyckoff'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_zona})
        continue
    ok_gat, lbl_gat, padroes_candle = guardiao_gatilho(df_w_norm)
    if not ok_gat:
        stats_filtros['bloqueios_gatilho'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_gat})
        continue
    ok_pay, lbl_pay, payoff_real = guardiao_payoff(entrada, alvo, stop_final, PARAMS_ATIVOS['custos_pct'])
    if not ok_pay:
        stats_filtros['bloqueios_payoff'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_pay})
        continue
    ok_corda, lbl_corda = guardiao_corda(df_w_norm, entrada, MODO_GEBRA, DIST_CORDA_MAX)
    if not ok_corda:
        stats_filtros['bloqueios_corda'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_corda})
        continue
    ok_macd, lbl_macd = guardiao_macd(df_w_norm, USAR_GUARDIAO_MACD)
    if not ok_macd:
        stats_filtros['bloqueios_macd'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_macd})
        continue
    ok_setor, lbl_setor, setor = guardiao_setor(ticker, contagem_setores, MAX_ATIVOS_POR_SETOR)
    if not ok_setor:
        stats_filtros['bloqueios_setor'] += 1
        status_ativos.append({'Ticker': ticker, 'Status': '❌ Recusado', 'Filtro': lbl_setor})
        continue
    
    contagem_setores[setor] = contagem_setores.get(setor, 0) + 1
    fat_liq = min(1.0, vol_fin / LIMITE_LIQUIDEZ_FINANCEIRA) if (pd.notna(vol_fin) and vol_fin and vol_fin > 0) else 0.5
    lote_base = int(risco_maximo / risco) if risco > 0 else 0
    lote_aj = max(1, int(lote_base * fat_liq))
    score_qualidade = calcular_score_qualidade({}, True, ell_valido, True, True, True, True, True, True, True, True, True, True, contexto_trap, is_arm_baixa, ef, regime_wyckoff)
    padroes = [k for k, v in padroes_candle.items() if v] if padroes_candle else []
    setup = {
        'Ticker': ticker, 'Modalidade': 'Swing', 'Direcao': 'COMPRA', 'Entrada': round(entrada, 2),
        'Método Stop': 'ATR+Swing', 'Stop Loss': round(stop_final, 2), 'Risco (R$)': round(risco, 2),
        'Alvo 3:1': round(alvo, 2), 'Alvo Recomendado': round(alvo, 2),
        'Resistência': round(rh, 2), 'Suporte': round(rl, 2), 'Regime': reg, 'Eficiência': ef,
        'Sentimento': 0, 'Volume Anormal': detectar_volume_anormal(df_w_norm),
        'Payoff Real': payoff_real, 'Lote': lote_aj,
        'Padrões Detectados': ', '.join(padroes) if padroes else 'Nenhum',
        'Score Qualidade': score_qualidade
    }
    stats_filtros['setup_aprovado_swing'] += 1
    oportunidades_swing.append(setup)
    status_ativos.append({'Ticker': ticker, 'Status': '✅ APROVADO', 'Filtro': 'Nenhum'})

logger.concluir_etapa("Análise de Setups", {'analisados': stats_filtros['total_analisados'], 'aprovados': stats_filtros['setup_aprovado_swing']})

if oportunidades_swing:
    oportunidades_swing = sorted(oportunidades_swing, key=lambda x: x.get('Score Qualidade', 0), reverse=True)[:MAX_SETUPS_POR_DIA]

logger.log(f"\n🎯 Swing: {len(oportunidades_swing)} | Position: {len(oportunidades_position)} setups", "RESULTADO")
logger.log(f"   Kelly: {kelly_pct*100:.2f}% | Regime: {regime_vol} | Modo: {MODO_GEBRA}", "RESULTADO")

enviar_log_sempre(logger, stats_filtros, status_ativos, tickers_liquidos, regime_vol, nh_nl, lad, oportunidades_swing, oportunidades_position, contagem_setores, kelly_pct, data_d, data_w, tickers_processados)

logger._flush_buffer()
logger.limpar_buffer()
logger.resumo_final()
logger.log("✅ Sistema v8.5.3 Production Ready concluído com sucesso", "SUCCESS")
print("\n✅ Execução concluída. Todas as correções aplicadas.")